# Chapter `1.2` - Tools

## Setup

### Module imports

In [1]:
from os import getenv
from dotenv import load_dotenv

from pprint import pprint
from typing import Dict, Any
from IPython.display import Markdown

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

from langchain.messages import AIMessage
from langchain.messages import HumanMessage

from tavily import TavilyClient

### **Gemini** model setup

In [25]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Integrating tools with agents
Providing a **system prompt** instructing the agent to make use of the tools provided for answering the user's prompts.

### Function tools

#### Tool definition
Using the `@tool` decorator for defining a **tool**.

In [23]:
@tool("square_root", description="Computes the square root of a given number.")
def tool_square_root(x: float) -> float:
    return x ** 0.5

In [18]:
@tool("square", description="Computes the square of a given number.")
def tool_square(x: float) -> float:
    return x ** 2

In [20]:
tool_square_root.invoke({"x": 344})

18.55

In [27]:
agent = create_agent(
    model=model,
    tools=[tool_square, tool_square_root],
    system_prompt="You are an arithmetic agent. Use the tools provided to you to calculate the square root and square of any number. Give the answer in relevant format."
)

In [ ]:
question = HumanMessage(content="What is the square of 121 and the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

Markdown(response['messages'][-1].content)

The square of 121 is 14641. The square root of 467 is approximately 21.61.

#### Full tracing - `FUNCTION`
Here, we can see what goes on _under the hood_ as a **Waterfall** diagram.
>
![Square prompt trace](../../pics/tool_trace_2.png)

### Web search tool

In [3]:
question2 = HumanMessage(content="How up to date is your training knowledge, Mr. Gemini?")
agent2 = create_agent(model=model)

response2 = agent2.invoke({
    "messages": [question2]
})

In [5]:
Markdown(response2['messages'][-1].content)

My knowledge cutoff is **June 2024**. Therefore, I cannot provide you with any information about events or developments that have occurred since that time.

> #### As we can see, for fetching the _latest information_ off of the web, we cannot rely on the model's knowledge base.

#### **Tavily** API setup
We can use this for enabling web search-based results from the LLM.

In [26]:
tavily_client = TavilyClient()

In [27]:
@tool("web_search_tool", description="Search the web for finding the most responses to the user's requests.")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query=query)

In [28]:
web_search_result = web_search.invoke("Which national team won the 2025 ICC Champions trophy?")
pprint(web_search_result['results'][0])

{'content': 'The Correct Answer is India. In News Champions Trophy 2025: India '
            'beats New Zealand by four wickets. Key Points India won the ICC '
            'Champions.',
 'raw_content': None,
 'score': 0.9999398,
 'title': '[Solved] Who won the ICC Champions Trophy 2025 Final?',
 'url': 'https://testbook.com/question-answer/who-won-the-icc-champions-trophy-2025-final--67ce56fd7d679ce5626620fd'}


In [31]:
agent3 = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="ALWAYS look for the correct answer using the web search tool provided to you. Look for the correct answer from the 'results' key-value pair from the web search results."
)

In [ ]:
question3 = "Who won the 2025 ICC Champions trophy?"
ws_response = agent3.invoke({"messages": [HumanMessage(content=question3)]})

print(ws_response['messages'][-1].content)

India won the 2025 ICC Champions Trophy, defeating New Zealand by four wickets in the final. This was India's fifth Champions Trophy final appearance and their third title win.


#### Full tracing - `WEB SEARCH`

![Tavily Tool Tracing](../../pics/tavily_tool_trace.png)